<a href="https://colab.research.google.com/github/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/blob/main/WHSAT_knowledge_graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install spacy networkx pandas
!python -m spacy download en_core_web_trf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 1.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/susara20010420/Workplace-Safety-Insights-from-the-Industrial-Safety-Health-Analytics-Dataset/main/cleaned_safety_data_set_A.csv")
df = df[['description', 'severity', 'critical_risk']].dropna()

In [3]:
import spacy

nlp = spacy.load("en_core_web_trf")

In [4]:
ENTITY_TYPES = [
    "TASK",
    "EQUIPMENT",
    "HAZARD",
    "CONTROL"
]

In [5]:
TASKS = ["welding", "cutting", "lifting", "drilling", "maintenance", "cleaning"]
EQUIPMENT = ["ladder", "forklift", "crane", "machine", "scaffold"]
HAZARDS = ["fall", "slip", "collision", "electric shock", "fire"]
CONTROLS = ["helmet", "gloves", "harness", "guardrail", "ppe"]

In [6]:
def extract_entities(text):
    doc = nlp(text.lower())

    entities = {
        "TASK": set(),
        "EQUIPMENT": set(),
        "HAZARD": set(),
        "CONTROL": set()
    }

    for token in doc:
        if token.text in TASKS:
            entities["TASK"].add(token.text)
        if token.text in EQUIPMENT:
            entities["EQUIPMENT"].add(token.text)
        if token.text in HAZARDS:
            entities["HAZARD"].add(token.text)
        if token.text in CONTROLS:
            entities["CONTROL"].add(token.text)

    return entities

In [7]:
kg_data = []

for i, row in df.iterrows():
    entities = extract_entities(row['description'])

    kg_data.append({
        "incident_id": i,
        "entities": entities,
        "severity": row['severity']
    })

In [8]:
import networkx as nx

G = nx.Graph()

for item in kg_data:
    incident = f"incident_{item['incident_id']}"
    G.add_node(incident, type="Incident", severity=item['severity'])

    for etype, values in item['entities'].items():
        for v in values:
            node_name = f"{etype}:{v}"
            G.add_node(node_name, type=etype)

            G.add_edge(incident, node_name, relation="involves")

In [9]:
for item in kg_data:
    entities = item['entities']

    for task in entities["TASK"]:
        for hazard in entities["HAZARD"]:
            G.add_edge(f"TASK:{task}", f"HAZARD:{hazard}", relation="causes")

    for hazard in entities["HAZARD"]:
        for control in entities["CONTROL"]:
            G.add_edge(f"HAZARD:{hazard}", f"CONTROL:{control}", relation="mitigated_by")

In [10]:
def detect_missing_control(entities):
    if "fall" in entities["HAZARD"] and "harness" not in entities["CONTROL"]:
        return 1
    return 0

df['missing_harness_flag'] = df['description'].apply(
    lambda x: detect_missing_control(extract_entities(x))
)

In [11]:
degree_centrality = nx.degree_centrality(G)

# Map back to dataframe
df['graph_degree'] = df.index.map(
    lambda i: degree_centrality.get(f"incident_{i}", 0)
)

In [12]:
nodes = []
edges = []

for n, data in G.nodes(data=True):
    nodes.append({
        "id": n,
        "type": data.get("type", "")
    })

for u, v, data in G.edges(data=True):
    edges.append({
        "source": u,
        "target": v,
        "relation": data.get("relation", "")
    })

pd.DataFrame(nodes).to_csv("kg_nodes.csv", index=False)
pd.DataFrame(edges).to_csv("kg_edges.csv", index=False)

In [14]:
pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 5.5 MB/s eta 0:00:00


In [17]:
from collections import defaultdict

hazard_counts = defaultdict(int)

# Iterate through all incident nodes in the graph
for node in G.nodes():
    if G.nodes[node]['type'] == 'Incident':
        # Find neighbors of the incident node
        for neighbor in G.neighbors(node):
            # Check if the neighbor is a HAZARD and if the edge relation is 'involves'
            if G.nodes[neighbor]['type'] == 'HAZARD' and G[node][neighbor]['relation'] == 'involves':
                hazard_counts[neighbor] += 1

# Sort hazards by frequency in descending order
sorted_hazards = sorted(hazard_counts.items(), key=lambda item: item[1], reverse=True)

print("Hazard Frequencies:")
for hazard, freq in sorted_hazards:
    print(f"{hazard}: {freq}")

Hazard Frequencies:
HAZARD:fall: 19
HAZARD:slip: 6
HAZARD:fire: 5
